In [8]:
import pandas as pd
import seaborn as sb
import numpy as np
import matplotlib.pyplot as mtb
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestRegressor
import joblib

In [9]:
%run preprocessing.ipynb

,stops,class,duration,days_left,departure_time_Afternoon,departure_time_Early_Morning,departure_time_Evening,departure_time_Late_Night,departure_time_Morning,departure_time_Night,...,arrival_time_Evening,arrival_time_Late_Night,arrival_time_Morning,arrival_time_Night,airline_AirAsia,airline_Air_India,airline_GO_FIRST,airline_Indigo,airline_SpiceJet,airline_Vistara
0,0,0,0.027347,0.0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,1,0
1,0,0,0.030612,0.0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
2,0,0,0.027347,0.0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0.028980,0.0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0.030612,0.0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300148,1,1,0.188776,1.0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1
300149,1,1,0.195714,1.0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
300150,1,1,0.265306,1.0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
300151,1,1,0.187143,1.0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1


<class 'pandas.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   stops                         300153 non-null  int64  
 1   class                         300153 non-null  int64  
 2   duration                      300153 non-null  float64
 3   days_left                     300153 non-null  float64
 4   departure_time_Afternoon      300153 non-null  int64  
 5   departure_time_Early_Morning  300153 non-null  int64  
 6   departure_time_Evening        300153 non-null  int64  
 7   departure_time_Late_Night     300153 non-null  int64  
 8   departure_time_Morning        300153 non-null  int64  
 9   departure_time_Night          300153 non-null  int64  
 10  arrival_time_Afternoon        300153 non-null  int64  
 11  arrival_time_Early_Morning    300153 non-null  int64  
 12  arrival_time_Evening          300153 non-null  int64  


In [10]:
X = y_encoded.values
Y = x['price'].values

In [11]:


target = 'price'

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    random_state=42,
    test_size=0.20
)
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_test : {y_test.shape}")


X_train : (240122, 22)
y_train : (240122,)
X_test : (60031, 22)
y_test : (60031,)


In [12]:
modeldumb = DummyRegressor(strategy='mean')

modeldumb.fit(X_train, y_train)

predictions = modeldumb.predict(X_test)

score = modeldumb.score(X_test, y_test)

mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

mape = mean_absolute_percentage_error(y_test, predictions)

print("MAPE:", mape * 100, "%")
print("R²:", score)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

MAPE: 238.20606742385294 %
R²: -5.741993791552602e-08
MAE: 19768.69424587037
MSE: 515482303.1678328
RMSE: 22704.23535747973


In [13]:
joblib.dump(modeldumb, '../models/modeldumb.joblib')

['../models/modeldumb.joblib']

In [14]:
LR_model = LinearRegression()

LR_model.fit(X_train, y_train)

y_pred = LR_model.predict(X_test)

score = LR_model.score(X_test, y_test)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

mape = mean_absolute_percentage_error(y_test, y_pred)

print("R²:", score)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAPE:", mape * 100, "%")

R²: 0.9067605218954887
MAE: 4532.140567650111
MSE: 48063298.15968857
RMSE: 6932.769876441059
MAPE: 42.39476022363981 %


In [15]:
joblib.dump(LR_model, '../models/LR_model.joblib')

['../models/LR_model.joblib']

In [16]:
X = y_encoded.values
Y = x['price'].values

k = 5

kf = KFold(
    n_splits=k,
    shuffle=True,
    random_state=42
)

FR_model = RandomForestRegressor(
    random_state=42
)

scores = cross_val_score(
    FR_model,
    X,
    Y,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

mae_scores = -scores

print("MAE for each fold:", mae_scores)
print("Average MAE:", np.mean(mae_scores))

MAE for each fold: [2403.80578989 2406.76938636 2396.19930708 2408.14430943 2387.41408456]
Average MAE: 2400.4665754659545


In [17]:
joblib.dump(FR_model, '../models/FR_model.joblib')

['../models/FR_model.joblib']